---
short_title: "Das Modul `csv`"
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# CSV einlesen mit dem `csv`-Modul

Im vorigen Kapitel haben wir gesehen, dass das manuelle Zerlegen mit `split()`
schnell an seine Grenzen stößt. Python wird mit dem Standardmodul `csv`
ausgeliefert, das {term}`CSV`-Dateien regelkonform einliest.

Das Modul müssen wir nicht installieren – es gehört zur
Standardbibliothek (standard library). Wir laden es mit `import csv`.

## `csv.reader`: Zeilen als Listen

`csv.reader` ist ein **Iterator** über die Zeilen der Datei. Jede Zeile wird als
Liste von Strings zurückgegeben.

In [ ]:
import csv

datei = "../assets/data/books_powerlaw_dataset.csv"

with open(datei, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    for nummer, zeile in enumerate(reader):
        if nummer > 3:
            break
        print(zeile)

```{note}
`newline=""` beim Öffnen verhindert, dass Python Zeilenumbrüche selbst
umwandelt. Das `csv`-Modul übernimmt die Behandlung der Zeilenenden dann
vollständig – das ist die offizielle Empfehlung.
```

```{note}
Einen **Iterator** können Sie sich vorerst wie eine besondere Liste vorstellen,
deren Inhalt Sie nicht einmal als ganzes sondern immer nur Element für Element
erhalten können. Dadurch können bspw. Daten verarbeitet werden, die größer sind
als der Arbeitsspeicher des Computers.
```

## Alle Zeilen einlesen

`list(reader)` liest alle Zeilen auf einmal. Danach trennen wir Header und
Datensätze.

In [2]:
with open(datei, encoding="utf-8", newline="") as f:
    alle_zeilen = list(csv.reader(f))

header = alle_zeilen[0]
daten = alle_zeilen[1:]

print("Header:", header)
print("Anzahl Datensätze:", len(daten))
print("Alle Zeilen mit 6 Feldern:", all(len(z) == 6 for z in daten))

Header: ['isbn', 'author', 'year', 'title', 'sales_year', 'sales']
Anzahl Datensätze: 180
Alle Zeilen mit 6 Feldern: True


Der Autor\*innenname `"Murakami, Haruki"` bleibt hier – anders als beim
manuellen Zerlegen – korrekt **ein** Feld:

In [3]:
print(daten[0])

['9780241951446', 'Murakami, Haruki', '1984', 'Norwegian Wood', '2020', '61']


## `csv.DictReader`: Zeilen als Wörterbücher

Mit `csv.DictReader` erhalten wir statt Listen **Wörterbücher
(dictionaries)**. Die Spaltennamen des Headers werden zu Schlüsseln. Das macht
den Code lesbarer, weil wir Spalten über ihren Namen ansprechen können.

In [4]:
with open(datei, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    print("Spaltennamen:", reader.fieldnames)

    for nummer, buch in enumerate(reader):
        if nummer >= 3:
            break
        print(buch["author"], "–", buch["title"])

Spaltennamen: ['isbn', 'author', 'year', 'title', 'sales_year', 'sales']
Murakami, Haruki – Norwegian Wood
Boltanski, Luc – Der neue Geist des Kapitalismus
Joyce, James – Ulysses


Jeder Datensatz ist ein `dict`. Auf einen einzelnen Wert greifen wir mit
`buch["sales"]` zu.

In [5]:
with open(datei, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    buch = next(reader)   # nur der erste Datensatz

print(buch)
print()
print("Titel:", buch["title"])
print("Jahr:", buch["year"])
print("Verkäufe:", buch["sales"])

{'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': '1984', 'title': 'Norwegian Wood', 'sales_year': '2020', 'sales': '61'}

Titel: Norwegian Wood
Jahr: 1984
Verkäufe: 61


## Trennzeichen und Anführungszeichen

CSV ist nicht genormt. Neben dem Komma sind das **Semikolon** (`;`) und der
**Tabulator** (`\t`) verbreitet. Das Trennzeichen steuern wir mit
`delimiter=...`, das Zeichen für die Feldeinfassung mit `quotechar=...`.

Zum Ausprobieren nutzen wir `io.StringIO`. Damit verhält sich ein String wie
eine Datei.

In [6]:
import io

semikolon_text = "titel;autor;jahr\nUlysses;Joyce, James;1922\nDer Prozess;Kafka, Franz;1925"

for zeile in csv.reader(io.StringIO(semikolon_text), delimiter=";"):
    print(zeile)

['titel', 'autor', 'jahr']
['Ulysses', 'Joyce, James', '1922']
['Der Prozess', 'Kafka, Franz', '1925']


Hier bleibt das Komma im Autorennamen erhalten, weil als Trennzeichen das
Semikolon vereinbart wurde. Genauso versteht `csv` Kommas **innerhalb** von
Anführungszeichen:

In [7]:
text_mit_quotes = "titel,autor\n\"Der Prozess, Roman\",Kafka\nUlysses,Joyce"

for zeile in csv.reader(io.StringIO(text_mit_quotes)):
    print(zeile)

['titel', 'autor']
['Der Prozess, Roman', 'Kafka']
['Ulysses', 'Joyce']


## Edge Cases: leere Zeilen und unbekannte Trennzeichen

**Leere Zeilen** liefert `csv.reader` als leere Liste `[]`. Beim Verarbeiten
sollten wir sie überspringen.

In [8]:
rohdaten = "isbn,titel\n1,Ulysses\n\n2,Der Prozess\n"

for zeile in csv.reader(io.StringIO(rohdaten)):
    if not zeile:          # leere Zeile überspringen
        continue
    print(zeile)

['isbn', 'titel']
['1', 'Ulysses']
['2', 'Der Prozess']


Ist das Trennzeichen unbekannt, kann `csv.Sniffer` es erraten. Das ist praktisch
für unbekannte Exporte, aber nicht immer zuverlässig.

In [9]:
beispiel = "isbn;titel;jahr\n9780141185422;Ulysses;1922\n"
dialekt = csv.Sniffer().sniff(beispiel)
print("Erkanntes Trennzeichen:", repr(dialekt.delimiter))

Erkanntes Trennzeichen: ';'


## Encoding

Das Encoding ist eine Eigenschaft der Datei, nicht des CSV-Formats. Es wird
deshalb an `open()` übergeben – nicht an `csv.reader`:

```python
with open(datei, encoding="utf-8", newline="") as f:
    ...
```

## Fehlende und überzählige Felder

`csv.DictReader` füllt fehlende Werte mit `None`. Überzählige Felder sammelt es
in einer Liste unter dem Schlüssel `restkey` (Standard: `None`). Beides lässt
sich gezielt steuern.

In [10]:
daten_mit_luecke = "a,b,c\n1,2\n3,4,5,6\n"

reader = csv.DictReader(io.StringIO(daten_mit_luecke), restkey="zu_viel")
for zeile in reader:
    print(zeile)

{'a': '1', 'b': '2', 'c': None}
{'a': '3', 'b': '4', 'c': '5', 'zu_viel': ['6']}


## Ausblick: Die „Pandas-Bridge“

Das Modul `csv` reicht für die meisten Aufgaben aus und erfordert keine
zusätzlichen Pakete. Für große Datenmengen mit Statistik und Visualisierung gibt
es die {term}`Bibliothek` **{term}`Pandas`**. Sie liest eine CSV-Datei mit einer einzigen Zeile:

```python
import pandas as pd

df = pd.read_csv("../assets/data/books_powerlaw_dataset.csv")
```

`pandas` führen wir erst in einem späteren Kapitel ein. Im nächsten Kapitel geht
es zunächst darum, wie wir eingelesene Daten auf Fehler prüfen.